In [ ]:
# --- STEP 1: IMPORT LIBRARIES ---
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense

In [ ]:
# --- STEP 2: LOAD YOUR DATA ---
from google.colab import files
upload = files.upload()
model_data = pd.read_csv("Model Data.csv")

Saving Model Data.csv to Model Data (1).csv


In [ ]:
# --- STEP 3: SPLIT INTO TRAIN AND TEST ---
test_size = 366
train_data = model_data[:-test_size].reset_index(drop=True)
test_data = model_data[-test_size:].reset_index(drop=True)

In [ ]:
# --- STEP 4: ENCODER SETUP ---
n_classes = 6  # Update as needed
category_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
category_encoder.fit(np.array(train_data['Response']).reshape(-1, 1))

def encode_categories(data):
    return category_encoder.transform(np.array(data['Response']).reshape(-1, 1))

categorical_ext_cols_model_1 = ['Season', 'Festivity', 'Daytype']
ext_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
ext_encoder.fit(train_data[categorical_ext_cols_model_1])

# --- STEP 5: SEQUENCE CREATION FUNCTION ---
def create_sequences(data, sequence_length=1, forecast_horizon=1,
                     categorical_ext_cols=None, numerical_ext_cols=None,
                     encode_external_categorical=False):
    X, y = [], []

    encoded_cat = encode_categories(data)

    if categorical_ext_cols and encode_external_categorical:
        encoded_ext_cat = ext_encoder.transform(data[categorical_ext_cols])
    else:
        encoded_ext_cat = np.empty((len(data), 0))

    if numerical_ext_cols:
        ext_num = data[numerical_ext_cols].to_numpy()
    else:
        ext_num = np.empty((len(data), 0))

    ext_all = np.hstack([encoded_ext_cat, ext_num])

    for i in range(len(data) - sequence_length - forecast_horizon + 1):
        cat_seq = encoded_cat[i:i + sequence_length]
        ext_seq = ext_all[i:i + sequence_length]
        full_seq = np.concatenate([cat_seq, ext_seq], axis=1)

        X.append(full_seq)
        y.append(data['Response'].iloc[i + sequence_length + forecast_horizon - 1])

    return np.array(X), np.array(y)

In [ ]:
# --- STEP 6: BUILD AND TRAIN LSTM ---
def build_model(input_shape):
    model = Sequential([
        Input(shape=input_shape),
        LSTM(64),
        Dense(n_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

# Train models with feature sets
# --- EXTERNAL FEATURE SETS ---
feature_sets = {
    'model_1': {
        'categorical': ['Season', 'Festivity', 'Daytype'],
        'numerical': [],
        'encode_cat': True
    },
    'model_2': {
        'categorical': [],
        'numerical': ['Season_sin', 'Season_cos', 'Diwali_sin', 'Diwali_cos', 'Daytype_sin', 'Daytype_cos'],
        'encode_cat': False
    }
}

sequence_length = 1
forecast_horizon = 1
models = {}

for model_name, features in feature_sets.items():
    print(f"\nTraining {model_name} with features: {features['categorical'] + features['numerical']}")

    X_train, y_train = create_sequences(
        train_data,
        sequence_length=sequence_length,
        forecast_horizon=forecast_horizon,
        categorical_ext_cols=features['categorical'],
        numerical_ext_cols=features['numerical'],
        encode_external_categorical=features['encode_cat']
    )

    input_shape = (X_train.shape[1], X_train.shape[2])
    model = build_model(input_shape)
    model.fit(X_train, y_train, epochs=10, batch_size=32, verbose=1)

    models[model_name] = model


Training model_1 with features: ['Season', 'Festivity', 'Daytype']
Epoch 1/10
57/57 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.4598 - loss: 1.6894
Epoch 2/10
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6334 - loss: 1.3171
Epoch 3/10
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6850 - loss: 1.0039
Epoch 4/10
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6897 - loss: 0.8630
Epoch 5/10
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7140 - loss: 0.7736
Epoch 6/10
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7165 - loss: 0.7431
Epoch 7/10
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7041 - loss: 0.7491
Epoch 8/10
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7089 - loss: 0.7451
Epoch 9/10
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7195 - loss: 0.7346
Epoch 10/10
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7112 - loss: 0.7281

Training model_2 with features: ['Season_sin', 'Season_cos', 'Diwali_sin', 'Diwali_cos', '

In [ ]:
# --- STEP 7: FORECAST ACCURACY FUNCTION ---
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score

def forecast_metrics(model, test_data, sequence_length=1, forecast_horizon=1,
                     categorical_ext_cols=None, numerical_ext_cols=None,
                     encode_external_categorical=False):
    true_labels = []
    pred_labels = []

    encoded_cat = encode_categories(test_data)

    if categorical_ext_cols and encode_external_categorical:
        encoded_ext_cat = ext_encoder.transform(test_data[categorical_ext_cols])
    else:
        encoded_ext_cat = np.empty((len(test_data), 0))

    if numerical_ext_cols:
        ext_num = test_data[numerical_ext_cols].to_numpy()
    else:
        ext_num = np.empty((len(test_data), 0))

    ext_all = np.hstack([encoded_ext_cat, ext_num])

    for i in range(len(test_data) - sequence_length - forecast_horizon + 1):
        cat_seq = encoded_cat[i:i + sequence_length]
        ext_seq = ext_all[i:i + sequence_length]
        full_seq = np.concatenate([cat_seq, ext_seq], axis=1)
        full_seq = full_seq.reshape(1, sequence_length, full_seq.shape[1])

        true_val = test_data['Response'].iloc[i + sequence_length + forecast_horizon - 1]
        pred = model.predict(full_seq, verbose=0)
        pred_label = np.argmax(pred)

        true_labels.append(true_val)
        pred_labels.append(pred_label)

    accuracy = accuracy_score(true_labels, pred_labels) * 100
    precision_macro = precision_score(true_labels, pred_labels, average='weighted', zero_division=0) * 100
    recall_macro = recall_score(true_labels, pred_labels, average='weighted', zero_division=0) * 100
    f1_macro = f1_score(true_labels, pred_labels, average='weighted', zero_division=0) * 100
    class_report = classification_report(true_labels, pred_labels, output_dict=True, zero_division=0)

    return {
        'accuracy': accuracy,
        'precision_macro': precision_macro,
        'recall_macro': recall_macro,
        'f1_macro': f1_macro,
        'per_class': class_report
    }

In [ ]:
def print_metrics(name, metrics):
    print(f"\n{name} Forecast Metrics:")
    print(f"Accuracy:               {metrics['accuracy']:.2f}%")
    print(f"Weighted Precision:     {metrics['precision_macro']:.2f}%")
    print(f"Weighted Recall:        {metrics['recall_macro']:.2f}%")
    print(f"Weighted F1 Score:      {metrics['f1_macro']:.2f}%")

metrics_1 = forecast_metrics(
    models['model_1'],
    test_data,
    sequence_length=sequence_length,
    forecast_horizon=forecast_horizon,
    categorical_ext_cols=feature_sets['model_1']['categorical'],
    numerical_ext_cols=feature_sets['model_1']['numerical'],
    encode_external_categorical=feature_sets['model_1']['encode_cat']
)

metrics_2 = forecast_metrics(
    models['model_2'],
    test_data,
    sequence_length=sequence_length,
    forecast_horizon=forecast_horizon,
    categorical_ext_cols=feature_sets['model_2']['categorical'],
    numerical_ext_cols=feature_sets['model_2']['numerical'],
    encode_external_categorical=feature_sets['model_2']['encode_cat']
)

print_metrics("Model 1", metrics_1)
print_metrics("Model 2", metrics_2)


Model 1 Forecast Metrics:
Accuracy:               72.88%
Weighted Precision:     72.92%
Weighted Recall:        72.88%
Weighted F1 Score:      72.89%

Model 2 Forecast Metrics:
Accuracy:               73.70%
Weighted Precision:     73.73%
Weighted Recall:        73.70%
Weighted F1 Score:      73.71%
